> Pin positions in recipes

In [ ]:
#| default_exp pin_map

In [ ]:
#| hide
%load_ext autoreload 
%autoreload 2

In [ ]:
#| export
from typing import List, Dict, Union
import numpy as np
from pathlib import Path
from cv_tools.core import *
import inspect
import cv2
from argparse import ArgumentParser
import matplotlib as mpl
from tqdm.auto import tqdm
import os
from fastcore.basics import *
from fastcore.all import *
from collections import Counter
import shutil
from PIL import Image

In [ ]:
from cv_tools.core import *

In [ ]:
#| export    
def get_full_image(
        img:np.ndarray,
        im_name:str,
        col_bbox:Dict[int, List[int]],
        recipe_no:str,
        save_path:Union[str, None]=None,
   ):
  'Get image of all the columns'

  num_cols_ = split_image_no()[recipe_no]



  pin_map_func =f'pin_map{recipe_no}'

  last_mask = np.zeros_like(img)
  pin_index_func=globals()[pin_map_func]
  all_cols = []
  num_cols = len(pin_index_func().keys())

  for col_no in range(num_cols):
        col_bbox_coord = col_bbox[col_no]

        col_data = img[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        pin_indexes = pin_index_func()[col_no]
        single_col_parts = split_image(col_data, num_cols_, 'vertical')
        parts=[]
        for idx, part in enumerate(single_col_parts):
            if idx in pin_indexes:
                part = part
                if save_path is not None:
                    cv2.imwrite(f'{save_path}/{im_name}_col_{col_no}_{idx}.png', part)
                #TODO: add pin mask
            else:
                part = np.zeros_like(part)
            parts.append(part)
        sn_col=np.concatenate(parts, axis=0)

        last_mask[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]] = sn_col
        all_cols.append(sn_col)
  full_part = np.concatenate(all_cols, axis=1)
  return last_mask, full_part


In [ ]:
#| export
def test(
        im_path:str,
        im_name:str,
        tmp_path:str,
        recipe_no:str,

    ):
    img = read_img(f'{im_path}/{im_name}')
    name_ = im_name
    pin_map_func = f'pin_map{recipe_no}'
    last_mask = np.zeros_like(img)
    pin_index_func=globals()[pin_map_func]
    all_cols=[]
    num_cols = len(pin_index_func().keys())
    bbox_func = globals()[f'get_all_columns_{recipe_no}']
    col_bbox = bbox_func(
                            img,
                            read_img(tmp_path)
                            )

    last_part, img_part = get_full_image(
        img=img,
        im_name=name_,
        col_bbox=col_bbox,
        recipe_no=recipe_no
        )
    show_(img_part)

In [ ]:
mpl.rcParams['image.cmap']='gray'

In [ ]:
#| include: false
#msk_path= Path(r'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/masks/sample_masks')
#im_path= Path(r'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/sample_images')
#msk_path.ls(),im_path.ls()
#com_images = L(set(get_name_(im_path.ls())).intersection(set(get_name_(msk_path.ls()))))
#len(im_path.ls())

In [ ]:
#| export
def split_image_no():
   return {
      '17':9,
      '18':11, 
      '19':11,
      '21':9,
      '25':11,
      '37':11,
      '43':11,
      '46':11,
      '47':11,
      '49':11,
      '50':11,
      '51':9,
      '55':9,
      '57':9,
      '68':11,
      '77':9,
      '79':11,
      '87':11,
      '88':9,
      '91':11,
      '92':11,
      '94':9,
      '96':11,
      '97':11,
      '103':11,
      '105':11,
      '108':9, 
      '122':11,
      '123':11,
      '125':9,
      '148': 11
      }

In [ ]:
#| export    
def get_full_image(
        img:np.ndarray,
        im_name:str,
        col_bbox:Dict[int, List[int]],
        recipe_no:str,
        save_path:Union[str, None]=None,
   ):
  'Get image of all the columns'

  num_cols_ = split_image_no()[recipe_no]



  pin_map_func =f'pin_map{recipe_no}'

  last_mask = np.zeros_like(img)
  pin_index_func=globals()[pin_map_func]
  all_cols = []
  num_cols = len(pin_index_func().keys())

  for col_no in range(num_cols):
        col_bbox_coord = col_bbox[col_no]

        col_data = img[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        pin_indexes = pin_index_func()[col_no]
        single_col_parts = split_image(col_data, num_cols_, 'vertical')
        parts=[]
        for idx, part in enumerate(single_col_parts):
            if idx in pin_indexes:
                part = part
                if save_path is not None:
                    cv2.imwrite(f'{save_path}/{im_name}_col_{col_no}_{idx}.png', part)
                #TODO: add pin mask
            else:
                part = np.zeros_like(part)
            parts.append(part)
        sn_col=np.concatenate(parts, axis=0)

        last_mask[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]] = sn_col
        all_cols.append(sn_col)
  full_part = np.concatenate(all_cols, axis=1)
  return last_mask, full_part


In [ ]:
#| export
def desired_im( 
              im_path:Union[str,Path],
              tmp_path:Union[str,Path],
              msk_path:Union[str, Path,None],
              img_sn_pin_save_path:Union[str, None]=None,
              msk_sn_pin_save_path:Union[str, None]=None,
              recipe_name:str='91',
                ):
    ''

    # reading all the images
    img = read_img(im_path, cv=True, gray=True) 
    tmp_img = read_img(tmp_path, cv=True, gray=True)
    if msk_path is not None:
        msk_img = read_img(msk_path, cv=True, gray=True)
    

    name_ = Path(im_path).name

    bbox_func = globals()[f'get_all_columns_{recipe_name}']


    col_bbox = bbox_func(
                                img,
                                tmp_img
                                )

    last_part, img_part = get_full_image(
        img=img,
        im_name=name_,
        save_path=img_sn_pin_save_path,
        col_bbox=col_bbox,
        recipe_no=recipe_name
        )
    if msk_path is not None:
        last_part, mask_part = get_full_image(
                                img=msk_img,
                                im_name=name_,
                                col_bbox=col_bbox,
                                recipe_no=recipe_name,
                                save_path=msk_sn_pin_save_path,
                                )
        return last_part, mask_part
    else:
        return last_part, img_part
    

   

In [ ]:
im_fn = 'VFV4.9.0.3_2024093011040764_ID_00394052563818896042438_In_148_FRONT_Missing Lead_image2.png'
im_path = Path(fr'M:\Fail_investigation\Missing_lead\ETPD_WAR_1_02_missing_20240813\missing_3109\missing_3109_unzip\main_im2_cropped\{im_fn}')

In [ ]:
temp_path = Path(r'E:\CurrentTrainingData20240812_trn_val\templates')
img = read_img(im_path)

In [ ]:
img = read_img(im_path)
recipe_no='148'
#show_(img[480:720, 460:720])
#cv2.imwrite(f'{temp_path}/{recipe_no}.png', img[480:720, 460:720])

In [ ]:
im_path

In [ ]:
recipe_no='148'
test(
im_path=im_path.parent,
im_name=im_fn,
tmp_path=f'{temp_path}/{recipe_no}.png', 
recipe_no='148')

In [ ]:
def pin_map148():
    return {
        0:[2, 3, 4,  6,7,8],
        1:[0, 10],
        2:[0, 5, 10],
        3:[],
        4:[],
        5:[0, 10 ],
        6:[4,  6 ],
        7:[],
        8:[0, 10],
        9:[0, 10],
        10:[0,10],
        11:[4, 5,6],
        12:[],
        13:[0, 10],
        14:[0, 10],
        15:[0, 1, 9, 10],
    }

def get_all_columns_148(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' Get bounding box for all columns in 57 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
   )
    x, y, w, h = bbox
    y0=135
    y1=1080
    x0 = x - 270
    c_d={}
    for i in range(16):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:

recipe_no='148'
test(
im_path=im_path.parent,
im_name=im_fn,
tmp_path=f'{temp_path}/{recipe_no}.png', 
recipe_no='148')

In [ ]:
#| export
def pin_map17(

   )->Dict:
   'Return recipe no 125 pin positions'
   return {
    0:[0, 8],
    1:[0, 8],
    2:[8],
    3:[0,8],
    4:[0,8],
    5:[3,8],
    6:[0,3,8],
    7:[0, 6, 8],
    8:[2, 4, 6, 8],
    9:[],
    10:[0,8],
   }

def get_all_columns_17(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' Get bounding box for all columns in 57 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=150
    y1=927
    x0 = x - 77
    c_d={}
    for i in range(11):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
#| export
def pin_map18(

) -> Dict:
   'Return recipe no 18 pin positions'
   return {
    0:[0, 1, 4, 7, 10],
    1:[0, 10],
    2:[10],
    3:[10],
    4:[0,10],
    5:[0,10],
    6:[0,10],
    7:[10],
    8:[10],
    9:[0,3, 7, 10],
    10:[0, 3, 10],
    11:[0],
    12:[5,6],
    13:[2,3],
    14:[0,7,10],
    15:[0,7,10]
   }

def get_all_columns_18(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' Get bounding box for all columns in 57 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=105
    y1=1070
    x0 = x - 264
    c_d={}
    for i in range(16):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
#| export
def pin_map19(

) -> Dict:
   'Return recipe no 19 pin positions'
   return {
    0:[0, 3, 10],
    1:[0, 5, 10],
    2:[0, 10],
    3:[10],
    4:[10],
    5:[0,10],
    6:[0,10],
    7:[0,10],
    8:[10],
    9:[10],
    10:[0, 4, 7,10],
    11:[0, 1, 4],
    12:[],
    13:[],
    14:[0,8,10],
    15:[0, 2, 3, 5, 6, 8,10]
   }

def get_all_columns_19(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' Get bounding box for all columns in 57 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=145
    y1=1085
    x0 = x - 530
    c_d={}
    for i in range(16):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
im2_img = Path(tst_im2_path, im2_name)
show_(im2_img)

In [ ]:
#| export
def frm_im2_im1_name(name_:str):
    return name_.replace('image2', 'image1')

In [ ]:
#| export
def frm_im1_im2_name(name_:str):
    return name_.replace('image1', 'image2')

In [ ]:
#| export
def frm_im2_pr_name(name_:str):
    return name_.replace('image2', 'processed_image')

In [ ]:
im2_name = frm_im1_im2_name(test_im_path.name)
im1_fn = test_im_path

In [ ]:
h, w = crop_img.shape[:2]
M_inv = cv2.getRotationMatrix2D(
                                (center_x, center_y),  180-c_ang, 1.0)
c_img = cv2.warpAffine(crop_img_r, M_inv, (w, h))

In [ ]:
show_(c_img)

In [ ]:
def find_rotation(
                 ref_image, 
                 tst_image):
    orb = cv2.ORB_create()

    kp1, des1 = orb.detectAndCompute(ref_image, None)
    kp2, des2 = orb.detectAndCompute(tst_image, None)
        # Detect keypoints and compute descriptors
    kp1, des1 = orb.detectAndCompute(ref_image, None)
    kp2, des2 = orb.detectAndCompute(tst_image, None)
    # Feature matching
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    
    

    # Match descriptors
    matches = bf.match(des1, des2, None)
    matches = sorted(matches, key=lambda x: x.distance)
    
   
    
    
    pt1 = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1,1,2)
    pt2 = np.float32([kp2[m.queryIdx].pt for m in matches]).reshape(-1,1,2)
    
    M, _ = cv2.estimateAffinePartial2D(pt1, pt2)
    #M, mask = cv2.findHomography(
    #pt2, 
    #pt1, 
    #cv2.RANSAC, 5.0)
    angles = []

    if M is not None:
        a_r = np.degrees(np.arctan2(
                         M[1,0],
                         M[0,0]))
        angles.append(a_r)
    angle = np.arctan2(M[0, 1], M[0, 0]) * 180 / np.pi
    return angle
    
    

In [ ]:
def back_rotate(
                
                test_img,
                c_ang):
    #h, w = ref_img.shape[:2]
    #center_x, center_y = h//2, w//2
    #M_inv = cv2.getRotationMatrix2D(
    #                            (center_x, center_y), -c_ang, 1.0)
    #c_img = cv2.warpAffine(tst_smart, M_inv, (w, h))
    rows, cols = test_img.shape[:2]
    M = cv2.getRotationMatrix2D((cols / 2, rows / 2), -angle, 1)
    rotated_img = cv2.warpAffine(test_img, M, (cols, rows))
    return rotated_img

In [ ]:
show_(ref_smart)

In [ ]:
show_(tst_img)

In [ ]:
ref_smart = read_img(Path(ref_im_path, 'ref_image_SMART.png'))
tst_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/main_im1')
tst_im_name = 'VFV4.9.0.0_2024051602265316_ID_01096043846818406192418_In_19_FRONT_Missing Lead_image1.png'
tst_img = read_img(Path(tst_path, tst_im_name))

In [ ]:
c_ang = find_rotation(
    ref_image=ref_smart,
    tst_image=tst_img
    )
print(c_ang)
f_test = back_rotate(tst_img, -(100-c_ang))
show_(f_test)

In [ ]:
c_ang = find_rotation(crop_img)

In [ ]:
M_inv = cv2.getRotationMatrix2D(
                                (center_x, center_y), -c_ang, 1.0)
c_img = cv2.warpAffine(tst_smart, M_inv, (w, h))

In [ ]:
show_(c_img)

In [ ]:
orb = cv2.ORB_create()

key1, des1 = orb.detectAndCompute(ref_smart, None)
key2, des2 = orb.detectAndCompute(tst_smart, None)
# Feature matching
matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = matcher.match(des1, des2)
matches = sorted(matches, key=lambda x: x.distance)
# Extract location of good points 
pt1 = np.zeros((len(matches), 2), dtype=np.float32)
pt2 = np.zeros((len(matches), 2), dtype=np.float32)

In [ ]:
for i, match in enumerate(matches):
    pt1[i,:] = key1[match.queryIdx].pt
    pt2[i,:] = key2[match.queryIdx].pt

In [ ]:
# Find homographay
H, mask = cv2.findHomography(pt1, pt2, cv2.RANSAC)

In [ ]:
if H is not None:
    rot_rad = np.arctan2(H[1,0], H[0,0])
    rot_deg = np.degrees(rot_rad)
    print(rot_deg)

In [ ]:
M_inv = cv2.getRotationMatrix2D((
                                 center_x, center_y, (w, h)))

In [ ]:
height, width = ref_smart.shape

In [ ]:
K = np.array([
    [width, 0, width/2],
    [0, height, height/2],
    [0, 0, 1]
])

In [ ]:
_, rot, trns, norm = cv2.decomposeHomographyMat(H,K)

In [ ]:
for r in rot:
    ang_det = np.degrees(np.arctan2(r[1,0], r[0,0]))
    print(ang_det)

In [ ]:
27-4

In [ ]:
d_comp = d_hom

In [ ]:
rot_mat = d_hom[1]
rot_mat[0][0]

In [ ]:
np.degrees(np.arctan2(rot_mat[0][0], rot_mat[1][0]))

In [ ]:
temp_path=Path(r'/home/ai_easypid.work/templates/templates')
temp_path.ls()

In [ ]:
cv2.imwrite(f'{temp_path}/smart_med.png', ref_smart[860:1155,920:1540])

In [ ]:
offset = 42
#cv2.imwrite(f'{temp_path}/smart_pin.png', ref_smart[410+2*offset:1547-offset,600+offset:2132-offset])

In [ ]:
x, y,w,h = get_template_part(
                           img=ref_smart,
                            tmp_img=read_img(f'{temp_path}/smart_pin.png')
)

In [ ]:
np.max(ref_smart)

In [ ]:
show_(ref_smart[y:y+h, x:x+w])

In [ ]:
show_(ref_smart)

In [ ]:
tst_im_path = Path(r'/home/ai_sintercra.work/Fail_investigation/VFV4.9.0.0_2024051622500965_ID_00492043846818406182418_In_19_FRONT_Missing Lead_image1.png')
tst_img = read_img(tst_im_path, cv=False).rotate(15)

In [ ]:
''

In [ ]:
rot_im1= rot_based_on_ref_img(
                     ref_img=ref_smart,
                     tst_img=tst_img, 
                    second_tst_img=None)

In [ ]:
show_(rot_im1)

In [ ]:

def rot_based_on_ref_img(
    ref_img:Union[np.ndarray, Image.Image], # Image will be used refeernece image
    tst_img:Union[np.ndarray, Image.Image], # Image where searched for key points
    second_tst_img:Union[np.ndarray, Image.Image]=None, # In case of another image rotation is necessary this image will be rotated same angle like tst_img
    show_m:bool=True # whether to show matching points of two images
    )->np.ndarray:
    'Rotate the tst_img based on ref image, find key points in ref image and then rotate tst_img'
    if isinstance(ref_img, Image.Image):
        ref_img = np.array(ref_img)
    
    if isinstance(tst_img, Image.Image):
        tst_img = np.array(tst_img)
    
    
    orb = cv2.ORB_create(50)
    # find the keypoints and descriptors with orb
    kp_t, des_t = orb.detectAndCompute(tst_img, None)  #kp1 --> list of keypoints
    kp_r, des_r = orb.detectAndCompute(ref_img, None)
    
    matcher = cv2.DescriptorMatcher_create(cv2.DESCRIPTOR_MATCHER_BRUTEFORCE_HAMMING)
    
    #Match descriptors.
    matches = matcher.match(des_r, des_t, None) 
    
    matches = sorted(matches, key = lambda x:x.distance)
    #Creates a list of all matches, just like keypoints
    img3 = cv2.drawMatches(ref_img, kp_r, tst_img, kp_t, matches[:10], None)
    if show_m: show_(img3)
    points_r = np.zeros((len(matches), 2), dtype=np.float32)  
    #Prints empty array of size equal to (matches, 2)
    points_t = np.zeros((len(matches), 2), dtype=np.float32)

    for i, match in enumerate(matches):
        points_r[i, :] = kp_r[match.queryIdx].pt    #gives index of the descriptor in the list of query descriptors
        points_t[i, :] = kp_t[match.trainIdx].pt    #gives index of the descriptor in the list of train descriptors
    M, mask = cv2.findHomography(points_r, points_t, cv2.RANSAC)

    if len(tst_img.shape) > 2:
        height, width, ch = ref_img.shape
    else:
        height, width = ref_img.shape
    
    im1Reg = cv2.warpPerspective(tst_img, M, (width, height)) 
    if second_tst_img is not None:
        im2Reg = cv2.warpPerspective(second_tst_img, M, (width, height))
        return im1Reg, im2Reg
    else:
        return im1Reg, mask
    

In [ ]:
show_(tst_img)

In [ ]:
tst_img.shape

In [ ]:
isinstance(tst_img, Image.Image)

In [ ]:
show_(tst_img)

In [ ]:
tst_img

In [ ]:
from skimage.color import rgb2gray

In [ ]:
ref_smart.shape,tst_img.shape
orb = cv2.ORB_create(50)

kp_t, des_t = orb.detectAndCompute(tst_img, None)
kp_r, des_r = orb.detectAndCompute(ref_smart, None)

In [ ]:
orb = cv2.ORB_create(50)
kp1, des1 = orb.detectAndCompute(ref_smart.astype(np.uint8), None)
kp2, des2 = orb.detectAndCompute(tst_img.astype(np.uint8), None)
matcher = cv2.DescriptorMatcher_create(cv2.DESCRIPTOR_MATCHER_BRUTEFORCE_HAMMING)

# Match descriptors.
matches = matcher.match(des1, des2, None)  #Creates a
matches = sorted(matches, key = lambda x:x.distance)
# Sort them in the order of their distance.
matches = sorted(matches, key = lambda x:x.distance)
img3 = cv2.drawMatches(im1,kp1, im2, kp2, matches[:10], None)
show_(img3)

In [ ]:
rot_im1, mask= rot_based_on_ref_img(
                     ref_img=ref_smart,
                     tst_img=tst_img.astype(np.uint8), 
                    second_tst_img=None)

In [ ]:
show_(center_crop(Image.fromarray(ref_smart),desired_width=2932,desired_height=1352, width_offset=0))

In [ ]:
show_(ref_smart[])

In [ ]:
show_(rot_im1)

In [ ]:
from skimage import data
from skimage import transform
from skimage.feature import match_descriptors, ORB, plot_matches
from skimage.color import rgb2gray
import matplotlib.pyplot as plt

In [ ]:
import matplotlib.pyplot as plt

from skimage import data
from skimage import transform
from skimage.color import rgb2gray
from skimage.feature import match_descriptors, plot_matches, SIFT

In [ ]:
show_(ref_smart)

In [ ]:
tst_img = transform.rotate(ref_smart, 15)
descriptor_extractor = ORB(n_keypoints=200)

descriptor_extractor.detect_and_extract(ref_smart)
keypoints_r = descriptor_extractor.keypoints
descriptors_r = descriptor_extractor.descriptors
show_(tst_img)

In [ ]:
descriptor_extractor.detect_and_extract(tst_img)
keypoints_t = descriptor_extractor.keypoints
descriptors_t = descriptor_extractor.descriptors

In [ ]:
matches12 = match_descriptors(
    descriptors_r, descriptors_t, max_ratio=0.6, cross_check=True
)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(11, 8))
plot_matches(ax, ref_smart, tst_img, keypoints_r, keypoints_t, matches12)

In [ ]:
M, mask = cv2.findHomography(keypoints_r, keypoints_t, cv2.RANSAC)
im2Reg = cv2.warpPerspective(ref_smart, M, (tst_img.shape[0], tst_img.shape[1]))

In [ ]:
show_(im2Reg)

In [ ]:
transform.AffineTransform(
                          tst_img, 
                           )

In [ ]:
matches12

In [ ]:
ax[0, 0].axis('off')
ax[0, 0].set_title("Original Image vs. Flipped Image\n" "(all keypoints and matches)")

plot_matches(ax[1], ref_smart, tst_img, keypoints_r, keypoints_t, matches12)
ax[1, 0].axis('off')
ax[1, 0].set_title(
    "Original Image vs. Transformed Image\n" "(all keypoints and matches)"
)

In [ ]:
show_(rot_im2)

In [ ]:
#for i in ref_im_path.ls():
    #if '1B' in Path(i).name:
        #stem = Path(i).stem
        #img_ = Image.open(i)
        
        #i#m = np.array(img_)
        #print(im.shape)
        #if len(im.shape) > 2:
            im = im[:,:,0]
        #cv2.imwrite(f'{i.parent}/{stem}_image1.png', im)
        #img = read_tif_img(f'{i}')
        

In [ ]:
img_

In [ ]:
#| export 
def pin_map21(

   )->Dict:
   '1B Return recipe no 51 pin positions'
   return {
    0:[0,3,4, 8],
    1:[0, 8],
    2:[],
    3:[],
    4:[0,5, 8],
    5:[0,5],
    6:[0,8],
    7:[],
    8:[5],
    9:[0,5],
    10:[0,7,8],
   }

def get_all_columns_21(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' 1B Get bounding box for all columns in 51 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=93
    y1=880
    x0 = x - 88
    c_d={}
    for i in range(11):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
temp_path = Path(r'/home/ai_easypid.work/templates/templates')

In [ ]:
recipe_no='21'
im_name='81651321_VFV4.1.5.2_2023011718571049_ID_00057038466816513212301_In_21_r_1_FRONT_Pass_image1.png'
im_path = Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/{recipe_no}')
img = read_img(f'{im_path}/{im_name}')
show_(img[350:610, 370:630])

In [ ]:
test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
recipe_no='25'
im_name='VFV4.1.8.0_2023021810285501_ID_00967033513816657262306_In_25_r_1_FRONT_Missing Lead-Img_02.png'
im_path = Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/{recipe_no}')
img = read_img(f'{im_path}/{im_name}')
#cv2.imwrite(f'{temp_path}/{recipe_no}.png',img[410:640,510:740
          #])

In [ ]:
#| export
def pin_map25(

   )->Dict:
   '2B Return recipe no 46 pin positions'
   return {
    0:[0, 4,5,10],
    1:[10],
    2:[],
    3:[10],
    4:[0,10],
    5:[10],
    6:[0,10],
    7:[0],
    8:[0,9,10],
    9:[0,9,10],
    10:[10],
    11:[],
    12:[10],
    13:[10],
    14:[10],
    15:[0,2,4,5,10],
   }

def get_all_columns_25(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' 2B Get bounding box for all columns in 46 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=50
    y1=1005
    x0 = x - 270
    c_d={}
    for i in range(16):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
recipe_no='25'
name_ = im_name
tmp_path = f'{temp_path}/{recipe_no}.png'
num_cols_ = split_image_no()[recipe_no]
pin_map_func =f'pin_map{recipe_no}'
last_mask = np.zeros_like(img)
pin_index_func=globals()[pin_map_func]
all_cols = []
num_cols = len(pin_index_func().keys())
print(f'Number of columns: {num_cols}')

bbox_func = globals()[f'get_all_columns_{recipe_no}']


col_bbox = bbox_func(
                            img,
                            read_img(tmp_path)
                            )
last_part, img_part = get_full_image(
    img=img,
    im_name=name_,
    col_bbox=col_bbox,
    recipe_no=recipe_no
    )
show_(img_part)

In [ ]:
#| export
def pin_map37(

   )->Dict:
   '2B Return recipe no 37 pin positions'
   return {
    0:[0, 1,2,3,4,5,6,7,8,9,10],
    1:[0,3,6,10],
    2:[],
    3:[],
    4:[],
    5:[],
    6:[10],
    7:[6],
    8:[6],
    9:[6,10],
    10:[6],
    11:[],
    12:[],
    13:[],
    14:[0,6,10],
    15:[0,1,2,4,5,6,8,9,10],
   }

def get_all_columns_37(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' 2B Get bounding box for all columns in 37 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=104
    y1=1055
    x0 = x - 260
    c_d={}
    for i in range(16):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
#| export
def pin_map43(

   )->Dict:
   '2B Return recipe no 43 pin positions'
   return {
    0:[0,4,5,6,7,10],
    1:[0,10],
    2:[],
    3:[],
    4:[8],
    5:[8],
    6:[3,4],
    7:[0,7,10],
    8:[0,7,10],
    9:[3,4],
    10:[8],
    11:[8],
    12:[],
    13:[],
    14:[0,10],
    15:[0,4,5,6,7,10],
   }

def get_all_columns_43(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' 2B Get bounding box for all columns in 37 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=52
    y1=1015
    x0 = x - 265
    c_d={}
    for i in range(16):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
recipe_no='43'
im_path = Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/{recipe_no}')
im_name='VFV4.1.8.0_2023020700072739_ID_00736042598816598032304_In_43_r_1_FRONT_Missing Lead-Img_02.png'

test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
#| export
def pin_map46(

   )->Dict:
   'Return recipe no 46 pin positions'
   return {
    0:[0, 9,10],
    1:[0],
    2:[6,10],
    3:[10],
    4:[0,1, 3,7],
    5:[0,7,10],
    6:[0,1,3,10],
    7:[6],
    8:[0,10],
    9:[0,8,10],
    10:[1,8],
    11:[0,10],
    12:[3,10],
    13:[6],
    14:[0,10],
    15:[0,10],
   }

def get_all_columns_46(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' Get bounding box for all columns in 46 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=102
    y1=1062
    x0 = x - 270
    c_d={}
    for i in range(16):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
#| export
def pin_map47(

   )->Dict:
   'Return recipe no 46 pin positions'
   return {
    0:[0, 6,7,10],
    1:[0,10],
    2:[],
    3:[],
    4:[],
    5:[],
    6:[3,4],
    7:[0,7,10],
    8:[0,7,10],
    9:[3,4],
    10:[],
    11:[],
    12:[],
    13:[],
    14:[0,10],
    15:[0,6,7,10],
   }

def get_all_columns_47(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' Get bounding box for all columns in 46 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=55
    y1=1010
    x0 = x - 275
    c_d={}
    for i in range(16):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
im_path=Path(r'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/47')
im_name='VFV4.1.5.2_2023013119244129_ID_00142042597816589102304_In_47_r_1_FRONT_Flying Lead-Img_02.png'
recipe_no='47'
temp_path = Path(r'/home/ai_easypid.work/templates/templates')
test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
#| export
def pin_map49(

   )->Dict:
   '2BReturn recipe no 50 pin positions'
   return {
    0:[0, 2,3, 4,5, 7,9, 10],
    1:[0,7],
    2:[3,10],
    3:[10],
    4:[],
    5:[10],
    6:[5,9],
    7:[3],
    8:[10],
    9:[0,3,10],
    10:[0,3],
    11:[],
    12:[2],
    13:[2],
    14:[0,10],
    15:[0,3,4,6,7,9,10],
   }

def get_all_columns_49(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' 2B Get bounding box for all columns in 50 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=60
    y1=1010
    x0 = x - 270
    c_d={}
    for i in range(16):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
recipe_no='49'
im_path=Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/sample_images/{recipe_no}')
im_name='81646594_VFV4.1.5.2_2023010713135174_ID_00370042633816465942251_In_49_r_1_FRONT_Pass_image1_var_50.png'
temp_path = Path(r'/home/ai_easypid.work/templates/templates')
test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
#| export
def pin_map50(

   )->Dict:
   '2BReturn recipe no 50 pin positions'
   return {
    0:[0, 1, 4, 7, 10],
    1:[0, 10],
    2:[10],
    3:[10],
    4:[0,10],
    5:[0,10],
    6:[0,10],
    7:[10],
    8:[10],
    9:[0,3,7,10],
    10:[0,3,10],
    11:[0],
    12:[5,6],
    13:[2,3],
    14:[0,7,10],
    15:[0,7,10],
   }

def get_all_columns_50(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' 2B Get bounding box for all columns in 50 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=102
    y1=1065
    x0 = x - 260
    c_d={}
    for i in range(16):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
#cv2.imwrite(f'{temp_path}/{recipe_no}.png', img[450:700,500:750])

In [ ]:
#| export
def pin_map51(

   )->Dict:
   '1B Return recipe no 51 pin positions'
   return {
    0:[0, 8],
    1:[0, 8],
    2:[8],
    3:[0,8],
    4:[0,8],
    5:[3, 8],
    6:[0,3,8],
    7:[0,6,8],
    8:[2,4,6,8],
    9:[],
    10:[0,8],
   }

def get_all_columns_51(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' 1B Get bounding box for all columns in 51 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=150
    y1=930
    x0 = x - 87
    c_d={}
    for i in range(11):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
#| export
def pin_map55(

   )->Dict:
   ' 1B Return recipe no 55 pin positions'
   return {
    0:[0, 1,2, 3, 8],
    1:[0, 8],
    2:[8],
    3:[],
    4:[4],
    5:[3,5],
    6:[8],
    7:[0,5,8],
    8:[0],
    9:[0],
    10:[0,6,7,8],
   }

def get_all_columns_55(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' 1B Get bounding box for all columns in 55 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=150
    y1=927
    x0 = x - 97
    c_d={}
    for i in range(11):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
#| export
def pin_map57(

   )->Dict:
   'Return recipe no 57 pin positions'
   return {
    0:[0, 2, 3, 6, 8],
    1:[0, 8],
    2:[0,8],
    3:[],
    4:[5],
    5:[5],
    6:[],
    7:[5],
    8:[5],
    9:[0,8],
    10:[0,1, 3, 5,6,7,8],
   }

def get_all_columns_57(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' Get bounding box for all columns in 57 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=150
    y1=927
    x0 = x - 77
    c_d={}
    for i in range(11):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
#| export
def pin_map68():
  '2B Pin mapping for recipe 68'
  return {
        0:[0,1,2,3,4,5,6,7,8,9,10],
        1:[0, 3,6,10],
        2:[],
        3:[],
        4:[],
        5:[],
        6:[10],
        7:[6],
        8:[6],
        9:[6,10],
        10:[6],
        11:[],
        12:[],
        13:[],
        14:[0,6,10],
        15:[0,1,2,4,5,6,8,9,10]
        }

#| export
def get_all_columns_68(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  '2B based on template image get all the columns'
  bbox = get_template_part(
    tst_img,
    tmp_img
    )
  x, y, w, h = bbox
  y0=45
  y1=1015
  x0 = x - 278
  c_d={}
  for i in range(16):
    x1 = x0 + 87
    c_d[i] = [y0, y1, x0, x1]
    x0 = x1
  return c_d

In [ ]:
recipe_no='68'
im_path = Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/{recipe_no}')
im_name='VFV4.1.8.0_2023041112053597_ID_00241047605816849862312_In_68_r_1_FRONT_Missing Lead-Img_02.png'
temp_path = Path(r'/home/ai_easypid.work/templates/templates')
test(
    im_path=im_path,
    im_name=im_name,
    tmp_path=f'{temp_path}/{recipe_no}.png',
    recipe_no=recipe_no
)

In [ ]:
#| export
def pin_map77():
    '1B Pin mapping for recipe 77'
    return {
        0:[0,4,6,8],
        1:[0,8],
        2:[],
        3:[0,2],
        4:[0],
        5:[8],
        6:[0,2,4,6,8],
        7:[0],
        8:[],
        9:[0,2,8],
        10:[0,4,6,8],
        }

def get_all_columns_77(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' Get bounding box for all columns in 77 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=95
    y1=880
    x0 = x - 87
    c_d={}
    for i in range(11):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
recipe_no='77'
im_path = Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/{recipe_no}')
im_name='81643675_VFV4.1.5.2_2022123011255114_ID_00730043637816436752250_In_77_r_1_FRONT_Pass_image1.png'
temp_path = Path(r'/home/ai_easypid.work/templates/templates')
test(
    im_path=im_path,
    im_name=im_name,
    tmp_path=f'{temp_path}/{recipe_no}.png',
    recipe_no=recipe_no
)

In [ ]:
#| export
def pin_map79():
  '2B Pin mapping for recipe 79'
  return {
        0:[0,1,2,6,7,10],
        1:[0, 1,10],
        2:[4],
        3:[],
        4:[0,1,8,9],
        5:[],
        6:[0,4,5],
        7:[0],
        8:[],
        9:[4,5],
        10:[5],
        11:[0,1, 8,9],
        12:[5],
        13:[4],
        14:[0,1, 10],
        15:[0,1,2,6,7,10]
        }

#| export
def get_all_columns_79(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  '2B based on template image get all the columns'
  bbox = get_template_part(
    tst_img,
    tmp_img
    )
  x, y, w, h = bbox
  y0=102
  y1=1065
  x0 = x - 270
  c_d={}
  for i in range(16):
    x1 = x0 + 87
    c_d[i] = [y0, y1, x0, x1]
    x0 = x1
  return c_d

In [ ]:
#| export

def pin_map87():
  'Pin mapping for recipe 87'
  return {
        0:[3,4,10],
        1:[0, 10],
        2:[0,10],
        3:[],
        4:[0,10],
        5:[0,10],
        6:[4],
        7:[4,8],
        8:[6,7,8],
        9:[1,2],
        10:[6,10],
        11:[0,6,10],
        12:[10],
        13:[0],
        14:[0,10],
        15:[0,10]
        }

In [ ]:
#| export
def get_all_columns_87(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  'based on template image get all the columns'
  bbox = get_template_part(
    tst_img,
    tmp_img
    )
  x, y, w, h = bbox
  y0=102
  y1=1065
  x0 = x - 270
  c_d={}
  for i in range(16):
    x1 = x0 + 87
    c_d[i] = [y0, y1, x0, x1]
    x0 = x1
  return c_d

In [ ]:
import os 

In [ ]:
#| export
def pin_map88():
    '1B Pin mapping for recipe 88'
    return {
        0:[0,7,8],
        1:[2],
        2:[0],
        3:[0,8],
        4:[0,5],
        5:[0,5],
        6:[0,2,5],
        7:[0,8],
        8:[8],
        9:[0,2],
        10:[0],
        }

def get_all_columns_88(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' Get bounding box for all columns in 88 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=100
    y1=880
    x0 = x - 87
    c_d={}
    for i in range(11):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
#| export

def pin_map91():
  'Pin mapping for recipe 91'
  return {
        0:[0,10],
        1:[0, 6],
        2:[0,6,10],
        3:[3],
        4:[3,8,9,10],
        5:[0],
        6:[0,4,5,10],
        7:[0],
        8:[3,4,10],
        9:[],
        10:[0,3,4,8,9,10],
        11:[0,8,9,10],
        12:[],
        13:[0, 5,6,10],
        14:[0],
        15:[0,10]
        }

In [ ]:
#| export
def get_all_columns_91(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  'based on template image get all the columns'
  bbox = get_template_part(
    tst_img,
    tmp_img
    )
  x, y, w, h = bbox
  y0=62
  y1=1005
  x0 = x - 280
  c_d={}
  for i in range(16):
    x1 = x0 + 87
    c_d[i] = [y0, y1, x0, x1]
    x0 = x1
  return c_d

In [ ]:
recipe_no='91'
im_path = Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/{recipe_no}')
im_name='VFV4.1.8.0_2023101921351250_ID_00751047042817691762341_In_91_r_1_FRONT_Pass_image2_var_80.png'
temp_path = Path(r'/home/ai_easypid.work/templates/templates')
test(
    im_path=im_path,
    im_name=im_name,
    tmp_path=f'{temp_path}/{recipe_no}.png',
    recipe_no=recipe_no
)

In [ ]:
#| export
def pin_map92():
  'Pin mapping for recipe 92'
  return {
        0:[0,1,2, 3, 4, 5, 6,7,8,9,10],
        1:[0,3, 6,10],
        2:[],
        3:[],
        4:[],
        5:[],
        6:[10],
        7:[6],
        8:[6],
        9:[6,10],
        10:[6],
        11:[],
        12:[],
        13:[],
        14:[0,6,10],
        15:[0,1,2,4,5,6, 8,9,10]
        }

def get_all_columns_92(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  'based on template image get all the columns'
  bbox = get_template_part(
    tst_img,
    tmp_img
    )
  x, y, w, h = bbox
  y0=55
  y1=1010
  x0 = x - 275
  c_d={}
  for i in range(16):
    x1 = x0 + 87
    c_d[i] = [y0, y1, x0, x1]
    x0 = x1
  return c_d

In [ ]:
def pin_map94():
    '2B Pin mapping for recipe 94'
    return {
        0:[0,2,3,8],
        1:[0, 8],
        2:[0,8],
        3:[],
        4:[5],
        5:[5],
        6:[],
        7:[5],
        8:[5],
        9:[0,8],
        10:[0, 1, 3, 5, 6,7,8],
        }

def get_all_columns_94(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  '2B based on template image get all the columns'
  bbox = get_template_part(
  tst_img,
  tmp_img
  )
  x, y, w, h = bbox
  y0=135
  y1=920
  x0 = x - 87
  c_d={}
  for i in range(11):
      x1 = x0 + 87
      c_d[i] = [y0, y1, x0, x1]
      x0 = x1
  return c_d

In [ ]:
test(
    im_path=Path(fr'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped'),
    im_name='VFV4.7.9.5_2024031507332669_ID_00834046305818203612409_In_94_r_1_FRONT_Flying Lead_image2.png',
    recipe_no='94',
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
#| export
def pin_map96():
  'Pin mapping for recipe 96'
  return {
        0:[0,10],
        1:[0,10],
        2:[0,10],
        3:[10],
        4:[8,9,10],
        5:[],
        6:[0,10],
        7:[0,4,9,10],
        8:[1,2,4,9,10],
        9:[0,9,10 ],
        10:[0,5],
        11:[0,5,9,10],
        12:[10],
        13:[10],
        14:[0,10],
        15:[0,10]
        }

def get_all_columns_96(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  'based on template image get all the columns'
  bbox = get_template_part(
    tst_img,
    tmp_img
    )
  x, y, w, h = bbox
  y0=55
  y1=1010
  x0 = x - 265
  c_d={}
  for i in range(16):
    x1 = x0 + 87
    c_d[i] = [y0, y1, x0, x1]
    x0 = x1
  return c_d

In [ ]:
recipe_no='96'
im_path = Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/{recipe_no}')
im_name='VFV4.1.9.0_2023030109232925_In_96_old_and_new_with_new_ref_image._r_1_Additional Lead-Img_02.png'
test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
#| export
def pin_map97():
  'Pin mapping for recipe 97'
  return {
        0:[0,10],
        1:[0,10],
        2:[0,10],
        3:[10],
        4:[8,9,10],
        5:[],
        6:[0,9,10],
        7:[0,4,9,10],
        8:[1,2,4,9,10],
        9:[0,10 ],
        10:[0,5],
        11:[0,5,9,10],
        12:[10],
        13:[10],
        14:[0,10],
        15:[0,10]
        }

def get_all_columns_97(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  'based on template image get all the columns'
  bbox = get_template_part(
    tst_img,
    tmp_img
    )
  x, y, w, h = bbox
  y0=60
  y1=1000
  x0 = x - 280
  c_d={}
  for i in range(16):
    x1 = x0 + 87
    c_d[i] = [y0, y1, x0, x1]
    x0 = x1
  return c_d

In [ ]:
recipe_no='97'
im_path = Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/{recipe_no}')
im_name =im_name='81651162_VFV4.1.5.2_2023012208110731_ID_00711045421816511622301_In_97_r_1_FRONT_Pass_image1_var_50.png'

test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
#| export
def pin_map103():
  '2B Pin mapping for recipe 105'
  return {
        0:[0,2,6,7,9,10],
        1:[0],
        2:[0,10],
        3:[10],
        4:[0],
        5:[10],
        6:[0,10],
        7:[0,7,6],
        8:[9,10],
        9:[0,10],
        10:[0],
        11:[0,9,10],
        12:[10],
        13:[0],
        14:[0,10],
        15:[0,9,10]
        }

def get_all_columns_103(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  '2B based on template image get all the columns'
  bbox = get_template_part(
    tst_img,
    tmp_img
    )
  x, y, w, h = bbox
  y0=110
  y1=1055
  x0 = x - 265
  c_d={}
  for i in range(16):
    x1 = x0 + 87
    c_d[i] = [y0, y1, x0, x1]
    x0 = x1
  return c_d

In [ ]:
recipe_no='103'
temp_path='/home/ai_easypid.work/templates/templates'
im_path='/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_18_17_57_val_frGrnd0.9570_epoch_180.h5/passed/images'
im_name='VFV4.7.9.5_2024030113503666_ID_00053047657818136342407_In_103_r_1_FRONT_Flying Lead_image2.png'
img = read_img (f'{im_path}/{im_name}')
test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
#| export
def pin_map105():
  '2B Pin mapping for recipe 105'
  return {
        0:[0,4,5,9,10],
        1:[0,10],
        2:[0,10],
        3:[1,2],
        4:[0,10],
        5:[0,10],
        6:[10],
        7:[3,10],
        8:[6,7,9,10],
        9:[],
        10:[],
        11:[2,3,4],
        12:[9,10],
        13:[9,10],
        14:[],
        15:[0,3,9,10]
        }

def get_all_columns_105(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  '2B based on template image get all the columns'
  bbox = get_template_part(
    tst_img,
    tmp_img
    )
  x, y, w, h = bbox
  y0=55
  y1=1010
  x0 = x - 275
  c_d={}
  for i in range(16):
    x1 = x0 + 87
    c_d[i] = [y0, y1, x0, x1]
    x0 = x1
  return c_d

In [ ]:
recipe_no='105'
im_path=Path(r'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/105')
im_name='81641156_VFV4.1.5.2_2023012213081057_ID_00147047400816411562249_In_105_r_1_FRONT_Pass_image1_var_200.png'
img = read_img(f'{im_path}/{im_name}')
test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
#| export
def pin_map108():
    '1B Pin mapping for recipe 108'
    return {
        0:[0,1, 2,3, 5, 6,7,8],
        1:[],
        2:[],
        3:[8],
        4:[8],
        5:[],
        6:[],
        7:[4],
        8:[4],
        9:[0,1, 4, 7,8],
        10:[0, 1, 4, 7,8],
        }

def get_all_columns_108(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  '2B based on template image get all the columns'
  bbox = get_template_part(
  tst_img,
  tmp_img
  )
  x, y, w, h = bbox
  y0=160
  y1=935
  x0 = x - 77
  c_d={}
  for i in range(11):
      x1 = x0 + 87
      c_d[i] = [y0, y1, x0, x1]
      x0 = x1
  return c_d

In [ ]:
recipe_no='108'
im_path=Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/{recipe_no}')
im_name='VFV4.7.9.5_2024012615154593_ID_00897049764818011022402_In_108_r_1_FRONT_Additional Lead_image2.png'
test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
#| export
def pin_map122():
  'Pin mapping for recipe 122'
  return {
        0:[0,1,10],
        1:[3, 6],
        2:[3,6],
        3:[10],
        4:[1,7,8],
        5:[1, 10],
        6:[3, 6],
        7:[6,10],
        8:[1, 8],
        9:[1, 8],
        10:[3,6,10],
        11:[6],
        12:[1,10],
        13:[1,7,8],
        14:[10],
        15:[2, 3]
        }

def get_all_columns_122(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  'based on template image get all the columns'
  bbox = get_template_part(
    tst_img,
    tmp_img
    )
  x, y, w, h = bbox
  y0=44
  y1=1015
  x0 = x - 278
  c_d={}
  for i in range(16):
    x1 = x0 + 87
    c_d[i] = [y0, y1, x0, x1]
    x0 = x1
  return c_d

In [ ]:
recipe_no='122'
im_name='257caed5-VFV4.1.8.0_2023040708423218_ID_00506049486816839882312_In_122_r_1_FRON_lt5VkaE.png'
im_path=Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/{recipe_no}')
test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
#| export
def pin_map123():
  'Pin mapping for recipe 122'
  return {
        0:[0,3,4,10],
        1:[0,10],
        2:[10],
        3:[10],
        4:[0],
        5:[0, 10],
        6:[0,5,10],
        7:[],
        8:[],
        9:[0, 5,6],
        10:[0,9,10],
        11:[0,9,10],
        12:[],
        13:[],
        14:[0,9,10],
        15:[0,9,10]
        }

def get_all_columns_123(
    tst_img:np.ndarray,# opencv image
    tmp_img:np.ndarray # opencv image
    ):
  'based on template image get all the columns'
  bbox = get_template_part(
    tst_img,
    tmp_img
    )
  x, y, w, h = bbox
  y0=55
  y1=1010
  x0 = x - 275
  c_d={}
  for i in range(16):
    x1 = x0 + 87
    c_d[i] = [y0, y1, x0, x1]
    x0 = x1
  return c_d

In [ ]:
temp_path=Path(r'/home/ai_easypid.work/templates/templates')

In [ ]:
im_name='VFV4.1.8.0_2023052108041934_ID_00975049782817046882319_In_123_r_1_FRONT_Additional Lead-Img_02.png'
im_path = Path(r'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/123')
img = read_img(f'{im_path}/{im_name}')
recipe_no='123'
test(
    im_path=im_path,
    im_name=im_name,
    recipe_no='123',
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
#| export
def pin_map125():
  'Pin mapping for recipe 125'
  return {
        0:[0,1,4,5,6,8],
        1:[8],
        2:[],
        3:[0],
        4:[0,5,8],
        5:[5,8],
        6:[0],
        7:[0],
        8:[],
        9:[0,2,3,4,6,7],
        10:[0],
        }

def get_all_columns_125(
        tst_img:np.ndarray, #open cv image
        tmp_img:np.ndarray, #open cv image
        ):
    ' Get bounding box for all columns in 57 template'
    bbox = get_template_part(
    tst_img,
    tmp_img
    )
    x, y, w, h = bbox
    y0=110
    y1=890
    x0 = x - 87
    c_d={}
    for i in range(11):
        x1 = x0 + 87
        c_d[i] = [y0, y1, x0, x1]
        x0 = x1
    return c_d

In [ ]:
recipe_no='125'
im_path=Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/sample_images')
im_name='VFV4.6.8.0_2023102508393668_In_125_r_1_Additional_Lead_image1.png'
temp_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/templates')
test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
len(bad_masks)

In [ ]:
print('hl')

In [ ]:
msk_path =Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240209/masks')
bad_masks=[]
for i in tqdm(msk_path.ls(),total=len(msk_path.ls())):
    msk = read_img(i)
    n_min, n_max = np.min(msk), np.max(msk)

    if not (n_min == 0 and n_max == 255 and len(np.unique(msk)) ==2):
        print('yes')
        print(i)
        print(np.unique(msk))
        bad_masks.append(i)


In [ ]:
s_msk_path =Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/sample_masks')
s_im_path =Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/sample_images')
s_im_path.ls(), s_msk_path.ls()

In [ ]:

im_path=Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418/images')
mask_path=Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418/masks')
image_names = get_name_(im_path.ls())

len(image_names)
images_n_ms = list(filter(lambda x: Path(x).name not in image_names, mask_path.ls()))
#for i in images_n_ms:
    #Path(i).unlink()

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

In [ ]:
df = pd.DataFrame()
for idx, i in  enumerate(image_names):
    r_n = get_m_name(i)
    df.loc[idx, 'fn'] = Path(i).name
    df.loc[idx,'rn' ] = r_n

In [ ]:
df['rn'].value_counts()

In [ ]:
# Check the distribution of classes in the 'rn' column
class_counts = df['rn'].value_counts()

# Filter out classes with less than 2 members
valid_classes = class_counts[class_counts >= 2].index

# Filter the DataFrame to include only the valid classes
df_filtered = df[df['rn'].isin(valid_classes)]

# Split the filtered DataFrame into training and test sets
trn, tst = train_test_split(df_filtered, test_size=0.2, random_state=42, stratify=df_filtered['rn'])

In [ ]:
#im_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418_trn_val/images')
#msk_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418_trn_val/masks')
#trn_im_path = Path(Path(im_path).parent, 'trn_images')
#val_im_path = Path(Path(im_path).parent, 'val_images')

#trn_msk_path = Path(Path(im_path).parent, 'trn_masks')
#val_msk_path = Path(Path(im_path).parent, 'val_masks')

#trn_im_path.mkdir(exist_ok=True, parents=True)
#val_im_path.mkdir(exist_ok=True, parents=True)

#trn_msk_path.mkdir(exist_ok=True, parents=True)
#val_msk_path.mkdir(exist_ok=True, parents=True)

In [ ]:
#for i in tqdm(trn['fn'].values, total=len(trn['fn'].values)):
    #name_ = Path(i).name
    #shutil.copyfile(f'{im_path}/{i}', f'{trn_im_path}/{i}')
    #shutil.copyfile(f'{msk_path}/{i}', f'{trn_msk_path}/{i}')

In [ ]:
#for i in tqdm(tst['fn'].values, total=len(tst['fn'].values)):
    #name_ = Path(i).name
    #shutil.copyfile(f'{im_path}/{i}', f'{val_im_path}/{i}')
    #shutil.copyfile(f'{msk_path}/{i}', f'{val_msk_path}/{i}')

In [ ]:
trn_im_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418_trn_val/trn_images')
trn_msk_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418_trn_val/trn_masks')

val_im_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418_trn_val/val_images')
val_msk_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418_trn_val/val_masks')
trn_im_path.ls(), trn_msk_path.ls(), val_im_path.ls(), val_msk_path.ls()  

In [ ]:
gen_im_path=Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/additional/generated_images')
gen_msk_path=Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/additional/generated_masks')
gen_names_trn = get_name_(np.random.choice(gen_im_path.ls(),100))# gen_msk_path.ls()
for i in gen_names_trn:
    
    shutil.copy(
        f'{gen_im_path}/{i}',
        f'{val_im_path}/{i}'

    )

    shutil.copy(
        f'{gen_msk_path}/{i}',
        f'{val_msk_path}/{i}'

    )

In [ ]:
im_path=Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418/masks')

In [ ]:
from private_easy_pin_detection.core import *

In [ ]:
mod_names = ([get_m_name(f'{i}') for i in bad_masks])
Counter(mod_names)

In [ ]:
for i in tqdm(bad_masks, total=len(bad_masks)):
    img = read_img(i)
    new_img=np.where(img>1,255,0 )
    print(np.unique(img))
    print(np.unique(new_img))
    cv2.imwrite(f'{i}',new_img)

In [ ]:
recipe_no='125'
im_path=Path(fr'/home/ai_easypid.work/RecipeWiseCurrentTrainingData/images/{recipe_no}')
im_name='VFV4.6.8.0_2023102508393668_In_125_r_1_Additional_Lead_image1.png'
test(
    im_path=im_path,
    im_name=im_name,
    recipe_no=recipe_no,
    tmp_path=f'{temp_path}/{recipe_no}.png'
)

In [ ]:
tmp_path=Path(TMP_PATH, '125.png')

In [ ]:
im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new_model/time_16_49_30_val_frGrnd0.9569_epoch_159_additional_pins/images/VFV4.7.9.5_2024030214331964_ID_00123049543818112692406_In_125_r_1_FRONT_Additional Lead_image2.png')
recipe_no = get_m_name(Path(im_path).name)
img = read_img(im_path)
name_ = im_path.name
num_cols_ = split_image_no()[recipe_no]
pin_map_func =f'pin_map{recipe_no}'
last_mask = np.zeros_like(img)
pin_index_func=globals()[pin_map_func]
all_cols = []
num_cols = len(pin_index_func().keys())

bbox_func = globals()[f'get_all_columns_{recipe_no}']


col_bbox = bbox_func(
                            img,
                            read_img(tmp_path)
                            )

last_part, img_part = get_full_image(
    img=img,
    im_name=name_,
    col_bbox=col_bbox,
    recipe_no=recipe_no
    )
show_(img_part)

In [ ]:
#| export
def pin_map108(

   )->Dict:
   'Return recipe no 108 pin positions'
   return {
    0:[0,1,2,3,5,6,7, 8],
    1:[],
    2:[],#1
    3:[8],#1
    4:[8],
    5:[],
    6:[],
    7:[4],
    8:[4], #6
    9:[0,1,4,7,8],
    10:[0,1,4,7,8],
   }


def full_image_108(
    img:np.ndarray,
    col_bbox:Dict,
    num_cols:int=11,
   ):
    'get full image mask for 108 recipe'

    last_mask = np.zeros_like(img)
    # getting current function name
    fn = inspect.currentframe().f_code.co_name
    fn_ = fn.split('_')[-1]
    pin_map_func=f'pin_map{fn_}'

    pin_pos_func =globals()[pin_map_func]
    all_cols = []
    for col_no in range(num_cols):
        col_bbox_coord = col_bbox[col_no]
        col_data = img[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        pin_indexes = pin_pos_func()[col_no]
        single_col_parts = split_image(col_data, 9, 'vertical')
        parts=[]
        for idx, part in enumerate(single_col_parts):
            if idx in pin_indexes:
                part = part
                #TODO: add pin mask
            else:
                part = np.zeros_like(part)
            parts.append(part)
        sn_col=np.concatenate(parts, axis=0)

        last_mask[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]] = sn_col
        all_cols.append(sn_col)
    full_part = np.concatenate(all_cols, axis=1)
    return last_mask, full_part


def desired_im_108( 
                  img:np.ndarray,
                  tmp_img:np.ndarray,
                  msk_img:np.ndarray=None,
                  num_cols:int=11,
                ):
    'From tempalte part get the desired image'

    # getting current function name
    fn = inspect.currentframe().f_code.co_name
    fn_ = fn.split('_')[-1]

    # assingning function name
    all_cols_func = globals()[f'get_all_columns_{fn_}']

    col_bbox = all_cols_func(
                                img,
                                tmp_img
                                )

    func_name = globals()[f'full_image_{fn_}']
    last_part, img_part = func_name(
        img=img,
        col_bbox=col_bbox,
        num_cols=num_cols,
        )
    if msk_img is not None:
        last_part, mask_part = func_name(
                                img=msk_img,
                                col_bbox=col_bbox,
                                num_cols=num_cols,
                                )
        return last_part, mask_part
    else:
        return last_part, img_part
    

In [ ]:
#| export
def extract_pins(
        im_path:str, #folder where all the images are available 
        msk_path:str, # mask folder where all the masks are available
        tmp_path:str, # path to the template only path, here one file named recipe_name.png should be there
        recipe_name:str, # name of the recipe
        img_sn_pin_save_path:str, # path where the images with pins will be saved
        img_sn_pin_msk_save_path:str, # path where the images with pins and masks will be saved
        ):

    for i in Path(im_path).ls():
        if msk_path is not None:
            msk_name=f'{msk_path}/{Path(i).name}',
        else:
            msk_name=None
        tmp_img = read_img(f'{tmp_path}/{recipe_name}.png', gray=True, cv=True)

        desired_im(
            im_path=i,
            msk_path=msk_name,
            tmp_path=tmp_path,
        )

In [ ]:
#| export
def parse_args_():
    'get all the arguments'
    parser = ArgumentParser()
    parser.add_argument(
        '--im_path',
        help='folder where all the images are available',
    )
    parser.add_argument(    
        '--msk_path',
        help='mask folder where all the masks are available',
        type=str,
    )
    parser.add_argument(
        '--tmp_path',
        type=str,
        help='path to the template only path, here one file named recipe_name.png should be there',
    )
    parser.add_argument(
        '--recipe_name',
        type=str,
        help='name of the recipe',
    )
    parser.add_argument(
        '--img_sn_pin_save_path',
        type=str,
        default=None,
        help='path where the single pin images with pins will be saved',

    )
    parser.add_argument(
        '--msk_sn_pin_save_path',
        default=None,
        help='path where the single pin masks with pins will be saved',

    )

In [ ]:
#| export
def main():
    args = parse_args_()
    extract_pins(
        im_path=args.im_path,
        msk_path=args.msk_path,
        tmp_path=args.tmp_path,
        recipe_name=args.recipe_name,
        img_sn_pin_save_path=args.img_sn_pin_save_path,
        img_msk_pin_save_path=args.msk_sn_pin_save_path,
    )

In [ ]:
#| export
#if __name__ == '__main__':
   #main() 

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('02_pin_map.ipynb')